# Global Ecommerce Sales Analysis

This notebook performs exploratory data analysis on a global ecommerce sales dataset for an end-to-end business case project.

## Objectives
- Validate the dataset
- Create derived business metrics
- Analyse sales and profit trends over time
- Compare category, product, customer, and regional performance
- Investigate discounting and shipping cost effects on profitability
- Generate summary outputs for reporting and dashboarding

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load dataset

Read the ecommerce dataset and confirm that it loads successfully.

In [7]:
file_path = "../data_cleaned/global_ecommerce_sales_cleaned.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data_cleaned/global_ecommerce_sales_cleaned.csv'

## Initial inspection

Review the structure of the dataset, column names, and a preview of the first few rows.

In [ ]:
print("\nColumns:")
print(df.columns.tolist())

print("\nPreview:")
display(df.head())

## Data type preparation

Convert the order date to datetime format and create additional time-based fields for trend analysis.

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df["Order_Date"].dt.quarter

## Feature engineering

Create validation fields and derived business metrics:
- Expected total sales
- Sales match check
- Profit margin
- Shipping cost ratio

In [ ]:
df["Expected_Total_Sales"] = (
    df["Quantity"] * df["Unit_Price"] * (1 - df["Discount_Percent"] / 100)
).round(2)

df["Recorded_Total_Sales"] = df["Total_Sales"].round(2)
df["Sales_Match"] = df["Expected_Total_Sales"] == df["Recorded_Total_Sales"]

df["Profit_Margin"] = np.where(df["Total_Sales"] != 0, df["Profit"] / df["Total_Sales"], 0)
df["Shipping_Cost_Ratio"] = np.where(df["Total_Sales"] != 0, df["Shipping_Cost"] / df["Total_Sales"], 0)

## Data quality checks

Check data types, missing values, and the date range covered by the dataset.

In [ ]:
print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DATE RANGE ===")
print("Min Date:", df["Order_Date"].min())
print("Max Date:", df["Order_Date"].max())

## Numeric summary

Review summary statistics for the main numeric fields to understand the overall distribution of values.

In [ ]:
print("=== NUMERIC SUMMARY ===")
display(df.describe(include="all"))

## KPI overview

Calculate the main headline business performance indicators.

In [ ]:
total_sales = df["Total_Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_ID"].nunique()
avg_order_value = df["Total_Sales"].mean()
avg_profit_order = df["Profit"].mean()
avg_profit_margin = df["Profit_Margin"].mean()

print("=== KPI OVERVIEW ===")
print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Unique Orders: {total_orders:,}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Average Profit per Order: ${avg_profit_order:,.2f}")
print(f"Average Profit Margin: {avg_profit_margin:.2%}")

## Monthly trend analysis

Aggregate sales and profit by month, calculate month-over-month growth, and compute rolling averages to smooth volatility.

In [ ]:
monthly_trend = (
    df.groupby(df["Order_Date"].dt.to_period("M"))
      .agg({"Total_Sales": "sum", "Profit": "sum"})
      .reset_index()
)

monthly_trend["Order_Date"] = monthly_trend["Order_Date"].dt.to_timestamp()
monthly_trend["Sales_Growth_%"] = monthly_trend["Total_Sales"].pct_change() * 100
monthly_trend["Profit_Growth_%"] = monthly_trend["Profit"].pct_change() * 100
monthly_trend["Sales_Rolling_3M"] = monthly_trend["Total_Sales"].rolling(3).mean()
monthly_trend["Profit_Rolling_3M"] = monthly_trend["Profit"].rolling(3).mean()

print("=== MONTHLY TREND ===")
display(monthly_trend.head())

print("\n=== MONTHLY GROWTH RATES ===")
display(monthly_trend[["Order_Date", "Sales_Growth_%", "Profit_Growth_%"]].head(12))

### Interpretation
- Review the overall direction of sales and profit over time
- Compare whether profit rises at the same pace as sales
- Use rolling averages to reveal the underlying trend beneath monthly volatility

## Yearly trend analysis

Compare year-over-year sales and profit performance.

In [ ]:
yearly_trend = (
    df.groupby("Year")
      .agg({"Total_Sales": "sum", "Profit": "sum"})
      .reset_index()
)

yearly_trend["Sales_Growth_%"] = yearly_trend["Total_Sales"].pct_change() * 100
yearly_trend["Profit_Growth_%"] = yearly_trend["Profit"].pct_change() * 100

print("=== YEARLY TREND ===")
display(yearly_trend)

### Interpretation
- Compare year-over-year growth
- Assess whether profitability improved or weakened over time
- Check whether growth is steady or uneven

## Category performance

Evaluate how each product category contributes to revenue, profit, volume, and efficiency.

In [ ]:
category_perf = (
    df.groupby("Product_Category")
      .agg(
          Total_Sales=("Total_Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum"),
          Orders=("Order_ID", "count")
      )
)

category_perf["Profit_Margin"] = category_perf["Profit"] / category_perf["Total_Sales"]
category_perf["Sales_Share_%"] = category_perf["Total_Sales"] / total_sales * 100
category_perf["Profit_Share_%"] = category_perf["Profit"] / total_profit * 100
category_perf["Profit_per_Order"] = category_perf["Profit"] / category_perf["Orders"]

print("=== CATEGORY PERFORMANCE ===")
display(category_perf.sort_values("Total_Sales", ascending=False))

### Interpretation
- Identify categories with high sales but weak profit
- Compare category margins and contribution shares
- Use profit per order to highlight efficiency, not just scale

## Product analysis

Identify the top-performing and weakest-performing products by sales and profit.

In [ ]:
top_products_sales = df.groupby("Product_Name")["Total_Sales"].sum().sort_values(ascending=False).head(10)
top_products_profit = df.groupby("Product_Name")["Profit"].sum().sort_values(ascending=False).head(10)
bottom_products_profit = df.groupby("Product_Name")["Profit"].sum().sort_values().head(10)

print("=== TOP 10 PRODUCTS BY SALES ===")
display(top_products_sales)

print("\n=== TOP 10 PRODUCTS BY PROFIT ===")
display(top_products_profit)

print("\n=== BOTTOM 10 PRODUCTS BY PROFIT ===")
display(bottom_products_profit)

### Interpretation
- Compare top-selling products with top-profit products
- Identify low-value or weak-margin products for review

## Product sales concentration (Pareto preview)

Assess whether a small number of products accounts for a large share of revenue.

In [ ]:
product_sales = (
    df.groupby("Product_Name", as_index=False)["Total_Sales"]
      .sum()
      .sort_values("Total_Sales", ascending=False)
      .reset_index(drop=True)
)

product_sales["Cumulative_Sales"] = product_sales["Total_Sales"].cumsum()
product_sales["Cumulative_Sales_%"] = product_sales["Cumulative_Sales"] / product_sales["Total_Sales"].sum() * 100
product_sales["Product_Rank_%"] = (product_sales.index + 1) / len(product_sales) * 100

print("=== PRODUCT SALES PARETO PREVIEW ===")
display(product_sales.head(10))

## Regional performance

Compare sales, profit, shipping cost, and margins across regions.

In [ ]:
regional_perf = (
    df.groupby("Region")
      .agg(
          Total_Sales=("Total_Sales", "sum"),
          Profit=("Profit", "sum"),
          Shipping_Cost=("Shipping_Cost", "sum"),
          Orders=("Order_ID", "count")
      )
)

regional_perf["Profit_Margin"] = regional_perf["Profit"] / regional_perf["Total_Sales"]
regional_perf["Sales_Share_%"] = regional_perf["Total_Sales"] / total_sales * 100
regional_perf["Profit_Share_%"] = regional_perf["Profit"] / total_profit * 100
regional_perf["Profit_per_Order"] = regional_perf["Profit"] / regional_perf["Orders"]

print("=== REGIONAL PERFORMANCE ===")
display(regional_perf.sort_values("Total_Sales", ascending=False))

## Top countries by sales

Identify the strongest country-level markets by revenue and profit.

In [ ]:
country_perf = (
    df.groupby("Country")
      .agg(
          Total_Sales=("Total_Sales", "sum"),
          Profit=("Profit", "sum"),
          Orders=("Order_ID", "count")
      )
)

country_perf["Profit_Margin"] = country_perf["Profit"] / country_perf["Total_Sales"]

print("=== TOP 10 COUNTRIES BY SALES ===")
display(country_perf.sort_values("Total_Sales", ascending=False).head(10))

## Customer segment performance

Compare the value and efficiency of each customer segment.

In [ ]:
segment_perf = (
    df.groupby("Customer_Segment")
      .agg(
          Total_Sales=("Total_Sales", "sum"),
          Profit=("Profit", "sum"),
          Orders=("Order_ID", "count")
      )
)

segment_perf["Profit_Margin"] = segment_perf["Profit"] / segment_perf["Total_Sales"]
segment_perf["Sales_Share_%"] = segment_perf["Total_Sales"] / total_sales * 100
segment_perf["Profit_Share_%"] = segment_perf["Profit"] / total_profit * 100
segment_perf["Profit_per_Order"] = segment_perf["Profit"] / segment_perf["Orders"]

print("=== CUSTOMER SEGMENT PERFORMANCE ===")
display(segment_perf.sort_values("Total_Sales", ascending=False))

## Payment method analysis

Review how payment methods compare in terms of orders, sales, profit, and margin.

In [ ]:
payment_perf = (
    df.groupby("Payment_Method")
      .agg(
          Total_Sales=("Total_Sales", "sum"),
          Profit=("Profit", "sum"),
          Orders=("Order_ID", "count")
      )
)

payment_perf["Profit_Margin"] = payment_perf["Profit"] / payment_perf["Total_Sales"]
payment_perf["Sales_Share_%"] = payment_perf["Total_Sales"] / total_sales * 100
payment_perf["Profit_Share_%"] = payment_perf["Profit"] / total_profit * 100

print("=== PAYMENT METHOD ANALYSIS ===")
display(payment_perf.sort_values("Total_Sales", ascending=False))

## Discount analysis

Assess how discounts relate to profit and whether deeper discounting appears commercially effective.

In [ ]:
discount_profit_corr = df["Discount_Percent"].corr(df["Profit"])
print(f"Correlation between Discount Percent and Profit: {discount_profit_corr:.4f}")

In [ ]:
df["Discount_Bucket"] = pd.cut(
    df["Discount_Percent"],
    bins=[-1, 10, 20, 30],
    labels=["0-10%", "10-20%", "20-30%"]
)

discount_bucket_analysis = (
    df.groupby("Discount_Bucket", observed=False)
      .agg(
          Avg_Sales=("Total_Sales", "mean"),
          Avg_Profit=("Profit", "mean"),
          Order_Count=("Order_ID", "count")
      )
)

print("=== DISCOUNT BUCKET ANALYSIS ===")
display(discount_bucket_analysis)

### Interpretation
- Review whether heavier discounts are linked to weaker profitability
- Check whether higher discounts create enough sales uplift to justify lower profit

## Shipping cost analysis

Evaluate the effect of shipping cost and shipping intensity on profitability.

In [ ]:
shipping_profit_corr = df["Shipping_Cost"].corr(df["Profit"])
shipping_ratio_margin_corr = df["Shipping_Cost_Ratio"].corr(df["Profit_Margin"])

print(f"Correlation between Shipping Cost and Profit: {shipping_profit_corr:.4f}")
print(f"Correlation between Shipping Cost Ratio and Profit Margin: {shipping_ratio_margin_corr:.4f}")

### Interpretation
- Larger orders can absorb shipping costs better
- The shipping cost ratio is more informative than shipping cost alone
- A strong negative relationship suggests logistics inefficiency on low-value orders

## Loss-making orders

Count how many orders generate negative profit.

In [ ]:
negative_profit_orders = (df["Profit"] < 0).sum()
print("Negative profit orders:", negative_profit_orders)

## Sales and profit relationship

Measure the strength of the relationship between order sales value and profit.

In [ ]:
sales_profit_corr = df["Total_Sales"].corr(df["Profit"])
print(f"Correlation between Total Sales and Profit: {sales_profit_corr:.4f}")

## Correlation matrix

Explore the relationships between key numerical variables to identify major profit and margin drivers.

In [ ]:
corr_cols = [
    "Quantity", "Unit_Price", "Discount_Percent", "Total_Sales",
    "Shipping_Cost", "Profit", "Profit_Margin", "Shipping_Cost_Ratio"
]

corr_matrix = df[corr_cols].corr()

print("=== CORRELATION MATRIX ===")
display(corr_matrix)

## EDA summary: key business takeaways

This section summarises the main findings from the analysis for reporting and dashboarding.

In [ ]:
print("=== EDA SUMMARY: KEY BUSINESS TAKEAWAYS ===")
print("1. Review monthly and yearly trends to confirm whether sales growth is consistent.")
print("2. Compare category and regional sales against profit to identify weak-margin areas.")
print("3. Use sales share and profit share to find which groups truly drive performance.")
print("4. Evaluate whether certain customer segments generate sales at the expense of profitability.")
print("5. Use discount analysis to determine whether deeper discounts are reducing margins.")
print("6. Use shipping analysis to identify whether logistics costs are hurting profit.")
print("7. Focus on top-performing and bottom-performing products for action planning.")
print("8. Use Pareto analysis to check whether a small number of products drives most revenue.")

## Export summary tables

Save key analysis outputs for use in reporting, Power BI, or documentation.

## Conclusion

The business is profitable overall, but performance is uneven across categories, products, customers, and markets.

### Main conclusions
- Revenue is concentrated in Furniture and Technology
- Consumer is the most valuable segment
- Office Supplies is the weakest category
- Deep discounts reduce profitability
- Shipping intensity is a major margin pressure
- A relatively small group of products drives a large share of sales

These findings support recommendations around pricing, discount control, fulfilment optimisation, and stronger focus on high-performing products and customer groups.